# Chapter 11 Companion Notebook: Regularization: Student Performance

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch11_Regularization_Student_Performance.ipynb)

This notebook accompanies Chapter 11 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Student Performance: Regularization

### Use "Student performace.csv".
- sex: student's sex (binary: 'F' - female or 'M' - male)
- age: student's age (numeric: from 15 to 22)
- address: student's home address type (binary: 'U' - urban or 'R' - rural)
- famsize: family size (binary: 'LE3' - less than or equal to 3 or 'GT3' - greater than 3)
- Pstatus: parent's cohabitation status (binary: 'T' - living together or 'A' - apart)
- Medu: mother's education (numeric: 0 - none, 1 - primary education (4th grade), 2 - 5th to 9th grade, 3 - secondary education, or 4 - higher education)
- Fedu: father's education (numeric: 0 - none, 1 - primary education (4th grade), 2 - 5th to 9th grade, 3 - secondary education, or 4 - higher education)
- Mjob: mother's job (nominal: 'teacher', 'health' (healthcare-related), 'services' (e.g., administrative or police), 'at_home', or 'other')
- Fjob: father's job (nominal: 'teacher', 'health' (healthcare-related), 'services' (e.g., administrative or police), 'at_home', or 'other')
- reason: reason for choosing this school (nominal: 'home' (close to home), 'reputation' (school reputation), 'course' (course preference), or 'other')
- guardian: student's guardian (nominal: 'mother', 'father', or 'other')
- traveltime: home-to-school travel time (numeric: 1 - <15 min, 2 - 15 to 30 min, 3 - 30 min to 1 hour, or 4 - >1 hour)
- studytime: weekly study time (numeric: 1 - <2 hours, 2 - 2 to 5 hours, 3 - 5 to 10 hours, or 4 - >10 hours)
- failures: number of past class failures (numeric: n if 1 ≤ n < 3, else 4)
- schoolsup: extra educational support (binary: 'yes' or 'no')
- famsup: family educational support (binary: 'yes' or 'no')
- paid: extra paid classes within the course subject (Math or Portuguese) (binary: 'yes' or 'no')
- activities: extra-curricular activities (binary: 'yes' or 'no')
- nursery: attended nursery school (binary: 'yes' or 'no')
- higher: wants to pursue higher education (binary: 'yes' or 'no')
- internet: Internet access at home (binary: 'yes' or 'no')
- romantic: in a romantic relationship (binary: 'yes' or 'no')
- famrel: quality of family relationships (numeric: from 1 - very bad to 5 - excellent)
- freetime: free time after school (numeric: from 1 - very low to 5 - very high)
- goout: going out with friends (numeric: from 1 - very low to 5 - very high)
- Dalc: workday alcohol consumption (numeric: from 1 - very low to 5 - very high)
- Walc: weekend alcohol consumption (numeric: from 1 - very low to 5 - very high)
- health: current health status (numeric: from 1 - very bad to 5 - very good)
- absences: number of school absences (numeric: from 0 to 93)
- grade: final grade (numeric: from 0 to 20, output target)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy import arange
from sklearn import model_selection
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline

### Create dummy variables from categorical variables.

In [ ]:
df = pd.read_csv("Student performance.csv")
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
# Define categorical columns to be converted into dummy variables
categorical = ["sex", "address", "famsize", "Pstatus", "Mjob", "Fjob", "reason", "guardian",
    "schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]

# Create dummy variables with proper names and drop the first category in each variable
df = pd.get_dummies(df, columns=categorical, drop_first=True)
df.head()

### Define dependent and independent variables. Then, standardize the predictors.
- Dependent variable: grade
- Independent variables: all other variables.

In [ ]:
# Split the data into features and target variable
y = df.grade
x = df.drop(columns=["grade"])

# Feature scaling
x_std=StandardScaler().fit_transform(x)

## Ridge regression
### Using the standardized predictors, run ridge regression with a proper shrinkage parameter range. Plot the estimated coefficients as a function of the shrinkage parameter.

In [ ]:
# Define a range of shrinkage parameters
a1 = np.linspace(0.001, 5000, 1000)

# Store coefficients for each alpha
coef1 = []

# Run Ridge regression for each alpha and store coefficients
for i in a1:
    m1 = Ridge(alpha = i).fit(x_std, y)
    coef1.append(m1.coef_)
print(np.shape(coef1))  # [alpha, x_std]

# Convert to a NumPy array for plotting
coef1 = np.array(coef1)

 - linspace(a, b, c): Create c evenly spaced points on a linear scale between a and b

In [ ]:
# Plot the coefficients as a function of the extended shrinkage parameter range
plt.figure(figsize=(10, 6))
for i in range(x.shape[1]):
    plt.plot(a1, coef1[:, i], label=x.columns[i])

plt.xlabel("Shrinkage Parameter")
plt.ylabel("Coefficient Value")
plt.grid(True)

- The magnitude of the coefficients decreases as the penalty parameter increases.

#### `alpha` is a hyperparameter that controls the regularization (shrinkage) strength.
- Smaller alpha values allow coefficients to remain close to their ordinary least squares (OLS) values. Larger alpha values push more coefficients toward zero to avoid overfitting the data.
- The amount of shrinkage a coefficient experiences dep
    - Variables with larger raw coefficients (before regularization) tend to shrink more significantly because Ridge penalizes large coefficients more heavily to prevent overfitting.  
    - If a variable is highly correlated with other predictors, Ridge regression distributes the effect across correlated variables, reducing their individual coefficients. Conversely, independent predictors with strong effects retain their influence and shrink less.  
    - Strong predictors (high explanatory power) tend to retain larger coefficients and weak predictors (low explanatory power) shrink toward zero faster, as Ridge discourages unnecessary complexity.  

### Perform 5-fold cross-validated ridge regression with the same shrinkage parameter range, using standardized predictors. Find the best shrinkage parameter that minimizes MSE.

In [ ]:
# Ridge tuning with proper standardization inside cross-validation
ridge_pipe = make_pipeline(StandardScaler(), Ridge())
ridge_search = GridSearchCV(ridge_pipe, {'ridge__alpha': a1}, scoring='neg_mean_squared_error', cv=5)
ridge_search.fit(x, y)

best_alpha_ridge = ridge_search.best_params_['ridge__alpha']
best_alpha_ridge

`x_std = StandardScaler().fit_transform(x)` standardizes the full dataset before cross-validation, which leaks information from the validation folds into the training folds. Use a pipeline so scaling happens inside each fold.

`ridge = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha_ridge))` creates a pipeline containing two sequential steps. The first step, `StandardScaler()`, standardizes the predictor variables. The second step, `Ridge(alpha=best_alpha_ridge)`, fits a Ridge regression model using the optimal value of the regularization parameter (`alpha`) obtained from the grid search. Whenever the pipeline is fit, `StandardScaler()` is first fit using only the training data, and the resulting scaling parameters are then applied to the training data before fitting the Ridge model. During prediction or cross-validation, the same scaling parameters are automatically applied to new data before making predictions. This approach prevents information from the test or validation data from leaking into the training process.

## Lasso regresssion
### Using the standardized predictors, run the lasso regression with a proper range of shrinkage parameter. Plot the estimated coefficients as a function of the shrinkage parameter.

In [ ]:
# Define a range of shrinkage parameters
a2 = np.linspace(0.001, 2, 100)

# Store coefficients for each alpha
coef2 = []

# Run Rasso regression for each alpha and store coefficients
for i in a2:
    m3 = Lasso(alpha = i, max_iter=10000).fit(x_std, y)
    coef2.append(m3.coef_)
print(np.shape(coef2))  # [alpha, x_std]

# # Convert to a NumPy array for plotting
coef2 = np.array(coef2)

In [ ]:
# Plot the coefficients as a function of the extended shrinkage parameter range
plt.figure(figsize=(10, 6))
for i in range(x.shape[1]):
    plt.plot(a2, coef2[:, i], label=x.columns[i])

plt.xlabel("Shrinkage Parameter")
plt.ylabel("Coefficient")
plt.grid(True)

- Many coefficients become zero as the penalty parameter increases.

### Run the lasso regression using 5-fold cross validation with the same shrinkage parameter range, using standardized predictors. Find the best shrinkage parameter that minimizes MSE.
- alpha_: optimal value of penalization chosen by cross validation

In [ ]:
# Lasso tuning with proper standardization inside cross-validation

lasso_pipe = make_pipeline(StandardScaler(), Lasso(max_iter=10000))
lasso_search = GridSearchCV(lasso_pipe, {'lasso__alpha': a2}, scoring='neg_mean_squared_error', cv=5)
lasso_search.fit(x, y)

best_alpha_lasso = lasso_search.best_params_['lasso__alpha']
best_alpha_lasso

### Compare performance of OLS, Ridge, and Lasso in terms of RMSE.

In [ ]:
# Perform 5-fold cross-validation for OLS
ols = LinearRegression()
ols_mse = -cross_val_score(ols, x, y, scoring="neg_mean_squared_error", cv=5).mean()

# Perform 5-fold cross-validation for Ridge (using the best alpha from previous tuning)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha_ridge))
ridge_mse = -cross_val_score(ridge, x, y, scoring="neg_mean_squared_error", cv=5).mean()

# Perform 5-fold cross-validation for Lasso (using the best alpha from previous tuning)
lasso = make_pipeline(StandardScaler(), Lasso(alpha=best_alpha_lasso, max_iter=10000))
lasso_mse = -cross_val_score(lasso, x, y, scoring="neg_mean_squared_error", cv=5).mean()

# Print RMSE values for OLS, Ridge, and Lasso models
print(f"OLS MSE: {ols_mse:.2f}")
print(f"Ridge MSE: {ridge_mse:.2f}")
print(f"Lasso MSE: {lasso_mse:.2f}")

- Regularization improves performance: Both Ridge and LASSO outperform OLS by reducing overfitting.
- Ridge performs best in terms of MSE.
- OLS does not need standardization. OLS estimates are invariant to linear scaling of the predictors. If you standardize a predictor, the regression coefficient changes accordingly, but the fitted values, residuals, R2, MSE, and predictions remain the same (apart from tiny numerical differences).